In [1]:
import pandas as pd
import os

# 1. Define exact paths provided by the user
base_dir = "../data/Retail"
demand_file = os.path.join(base_dir, "Retail_Demand_Outlook_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2.xlsx")
budget_file = os.path.join(base_dir, "Household_Budget_Expenditures_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2_2.xlsx")

# 2. Load the data using openpyxl
print("Loading data...")
df_demand = pd.read_excel(demand_file, header=None, engine='openpyxl')
df_budget = pd.read_excel(budget_file, header=None, engine='openpyxl')

def extract_numbers_from_row(df, search_term):
    """
    Finds the row containing the search term, strips text/commas, 
    and returns a clean dictionary of the Current and Future values across the 3 rings.
    """
    # Find the row that contains the exact search term (case-insensitive)
    mask = df.apply(lambda row: row.astype(str).str.contains(search_term, case=False, regex=False).any(), axis=1)
    if not mask.any():
        print(f"Warning: '{search_term}' not found in the data.")
        return None
    
    row_data = df[mask].iloc[0]
    
    # Clean the row: remove commas and convert to pure numbers
    cleaned_row = row_data.astype(str).str.replace(',', '', regex=False).str.strip()
    numeric_values = pd.to_numeric(cleaned_row, errors='coerce').dropna().tolist()
    
    # Filter out percentages (like Tapestry Segments which are < 1.0)
    numeric_values = [n for n in numeric_values if n > 1.0]
    
    # Esri horizontally outputs: 0-2m(Current, Future), 2-3m(Current, Future), 3-5m(Current, Future)
    if len(numeric_values) >= 6:
        return {
            '0-2_Mile_Current': numeric_values[0],
            '0-2_Mile_Future': numeric_values[1],
            '2-3_Mile_Current': numeric_values[2],
            '2-3_Mile_Future': numeric_values[3],
            '3-5_Mile_Current': numeric_values[4],
            '3-5_Mile_Future': numeric_values[5],
        }
    return None

# ==========================================
# STEP 1: EXTRACT DEMOGRAPHICS
# ==========================================
# We use the Budget file because the demographics rows are cleaner
demographics = {}
for demo in ['Population', 'Households', 'Median Household Income']:
    data = extract_numbers_from_row(df_budget, demo)
    if data:
        demographics[demo] = data

df_demo = pd.DataFrame(demographics).T

# ==========================================
# STEP 2: EXTRACT RETAIL EXPENDITURES
# ==========================================
# Map the sub-categories into your main buckets
categories = {
    'NG&S': ['Food at Home', 'Personal Care Products', 'Housekeeping Supplies'],
    'F&B': ['Food Away from Home', 'Alcoholic Beverages'],
    'GAFO': ['Apparel and Services', 'Entertainment & Recreation', 'Household Furnishings and Equipment']
}

exp_results = []
for bucket, sub_cats in categories.items():
    for cat in sub_cats:
        data = extract_numbers_from_row(df_demand, cat)
        if data:
            exp_results.append({
                'Bucket': bucket,
                'Category': cat,
                '0-2_Mile_Current': data['0-2_Mile_Current'],
                '2-3_Mile_Current': data['2-3_Mile_Current'],
                '3-5_Mile_Current': data['3-5_Mile_Current'],
                '0-2_Mile_Future': data['0-2_Mile_Future'],
                '2-3_Mile_Future': data['2-3_Mile_Future'],
                '3-5_Mile_Future': data['3-5_Mile_Future'],
            })

df_expenditures = pd.DataFrame(exp_results)

# ==========================================
# STEP 3: CALCULATE SUPPORTABLE SF (GAP ANALYSIS)
# ==========================================
# Note: Since the Esri reports provided do not contain existing store sales (Supply), 
# we calculate the Gap by determining what percentage of local demand your project can capture.

benchmarks = {'NG&S': 550, 'F&B': 650, 'GAFO': 400} # Industry average sales needed per SF
capture_rate = 0.15 # Assuming your project captures 15% of the local 0-2 mile market

df_analysis = df_expenditures.copy()
# Calculate how many dollars you capture
df_analysis['Target Capture ($) (0-2m)'] = df_analysis['0-2_Mile_Current'] * capture_rate
df_analysis['Sales per SF Benchmark'] = df_analysis['Bucket'].map(benchmarks)

# Calculate the square footage supported by those dollars
df_analysis['Supportable SF (0-2m)'] = (df_analysis['Target Capture ($) (0-2m)'] / df_analysis['Sales per SF Benchmark']).round(0)

# ==========================================
# OUTPUT TO CONSOLE (with comma formatting for numbers)
# ==========================================
pd.set_option('display.float_format', '{:,.0f}'.format) # Format for comma as thousand separator

def format_columns_with_commas(df, cols):
    df_fmt = df.copy()
    for col in cols:
        if col in df_fmt.columns:
            df_fmt[col] = df_fmt[col].apply(lambda x: '{:,.0f}'.format(x) if pd.notnull(x) else x)
    return df_fmt

print("\n--- DEMOGRAPHIC BASE (CURRENT YEAR) ---")
demo_cols = ['0-2_Mile_Current', '2-3_Mile_Current', '3-5_Mile_Current']
print(format_columns_with_commas(df_demo, demo_cols)[demo_cols])

print("\n--- CATEGORY EXPENDITURE (CURRENT YEAR) ---")
cat_cols = ['0-2_Mile_Current', '2-3_Mile_Current', '3-5_Mile_Current']
print(format_columns_with_commas(df_expenditures, ['0-2_Mile_Current', '2-3_Mile_Current', '3-5_Mile_Current'])[['Bucket', 'Category', '0-2_Mile_Current', '2-3_Mile_Current', '3-5_Mile_Current']])

print("\n--- SUPPORTABLE SQUARE FOOTAGE CALCULATION (0-2 MILE RING | 15% CAPTURE) ---")
analysis_cols = ['0-2_Mile_Current', 'Target Capture ($) (0-2m)', 'Supportable SF (0-2m)']
print(format_columns_with_commas(df_analysis, analysis_cols)[['Bucket', 'Category'] + analysis_cols])

# Optional: Save to a new clean CSV for your proforma
# df_analysis.to_csv(os.path.join(base_dir, "Cleaned_Retail_Demand_Analysis.csv"), index=False)

Loading data...


/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")



--- DEMOGRAPHIC BASE (CURRENT YEAR) ---
                        0-2_Mile_Current 2-3_Mile_Current 3-5_Mile_Current
Population                        59,183           55,073          165,981
Households                        21,387           22,539           67,761
Median Household Income           68,567           74,545           89,712

--- CATEGORY EXPENDITURE (CURRENT YEAR) ---
  Bucket                    Category 0-2_Mile_Current 2-3_Mile_Current  \
0   NG&S                Food at Home      139,369,490       18,213,601   
1   NG&S      Personal Care Products       11,724,578        1,538,639   
2   NG&S       Housekeeping Supplies       16,023,778        2,094,376   
3    F&B         Food Away from Home       78,189,105       10,219,449   
4    F&B         Alcoholic Beverages       12,042,192        1,569,034   
5   GAFO        Apparel and Services       46,968,852        6,142,133   
6   GAFO  Entertainment & Recreation       71,766,922        9,351,036   

  3-5_Mile_Current  


In [2]:
df_expenditures.head(20)

,Bucket,Category,0-2_Mile_Current,2-3_Mile_Current,3-5_Mile_Current,0-2_Mile_Future,2-3_Mile_Future,3-5_Mile_Future
0,NG&S,Food at Home,"139,369,490","18,213,601","190,703,098","157,583,091","163,892,911","26,810,187"
1,NG&S,Personal Care Products,"11,724,578","1,538,639","15,947,467","13,263,217","13,679,762","2,267,705"
2,NG&S,Housekeeping Supplies,"16,023,778","2,094,376","21,983,362","18,118,154","18,896,574","3,086,788"
3,F&B,Food Away from Home,"78,189,105","10,219,449","106,323,914","88,408,554","91,338,829","14,985,085"
4,F&B,Alcoholic Beverages,"12,042,192","1,569,034","17,013,868","13,611,226","14,642,861","2,371,007"
5,GAFO,Apparel and Services,"46,968,852","6,142,133","64,283,981","53,110,985","55,237,789","9,046,192"
6,GAFO,Entertainment & Recreation,"71,766,922","9,351,036","100,145,018","81,117,958","86,193,633","13,951,385"


In [3]:
import pandas as pd
import os

# 1. Define paths
base_dir = "../data/Retail"
demand_file = os.path.join(base_dir, "Retail_Demand_Outlook_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2.xlsx")
budget_file = os.path.join(base_dir, "Household_Budget_Expenditures_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2_2.xlsx")

# 2. Load data
df_demand = pd.read_excel(demand_file, header=None, engine='openpyxl')
df_budget = pd.read_excel(budget_file, header=None, engine='openpyxl')

def extract_esri_values(df, search_term, is_expenditure=True):
    """
    Precisely extracts Current and Future values.
    Expenditures: 3-column blocks (Current, Future, Growth)
    Demographics: 2-column blocks (Current, Future)
    """
    mask = df.apply(lambda row: row.astype(str).str.contains(search_term, case=False, regex=False).any(), axis=1)
    if not mask.any():
        return None
    
    row_data = df[mask].iloc[0]
    cleaned_row = row_data.astype(str).str.replace(',', '', regex=False).str.strip()
    nums = pd.to_numeric(cleaned_row, errors='coerce').dropna().tolist()
    
    # Filter out small percentages/indices (usually < 1.0 or very small)
    nums = [n for n in nums if n > 1.0]

    if is_expenditure:
        # Expenditure blocks: [R1_Curr, R1_Fut, R1_Grw, R2_Curr, R2_Fut, R2_Grw, R3_Curr, R3_Fut, R3_Grw]
        # We target indices: 0, 1, 3, 4, 6, 7
        if len(nums) >= 8:
            return {
                'Curr_0_2': nums[0], 'Fut_0_2': nums[1],
                'Curr_2_3': nums[3], 'Fut_2_3': nums[4],
                'Curr_3_5': nums[6], 'Fut_3_5': nums[7]
            }
    else:
        # Demographic blocks: [R1_Curr, R1_Fut, R2_Curr, R2_Fut, R3_Curr, R3_Fut]
        # We target indices: 0, 1, 2, 3, 4, 5
        if len(nums) >= 6:
            return {
                'Curr_0_2': nums[0], 'Fut_0_2': nums[1],
                'Curr_2_3': nums[2], 'Fut_2_3': nums[3],
                'Curr_3_5': nums[4], 'Fut_3_5': nums[5]
            }
    return None

# ==========================================
# STEP 1: DEMOGRAPHICS (2-column mapping)
# ==========================================
demo_results = {}
for item in ['Population', 'Households', 'Median Household Income']:
    data = extract_esri_values(df_budget, item, is_expenditure=False)
    if data: demo_results[item] = data

df_demo = pd.DataFrame(demo_results).T

# ==========================================
# STEP 2: EXPENDITURES (3-column mapping)
# ==========================================
categories = {
    'NG&S': ['Food at Home', 'Personal Care Products', 'Housekeeping Supplies'],
    'F&B': ['Food Away from Home', 'Alcoholic Beverages'],
    'GAFO': ['Apparel and Services', 'Entertainment & Recreation']
}

exp_list = []
for bucket, sub_cats in categories.items():
    for cat in sub_cats:
        data = extract_esri_values(df_demand, cat, is_expenditure=True)
        if data:
            data.update({'Bucket': bucket, 'Category': cat})
            exp_list.append(data)

df_exp = pd.DataFrame(exp_list)

# ==========================================
# OUTPUT VALIDATION
# ==========================================
pd.set_option('display.float_format', '{:,.0f}'.format)
print("--- VERIFIED HOUSEHOLD COUNTS ---")
print(df_demo.loc[['Households']])

print("\n--- VERIFIED EXPENDITURES (2-3 MILE CORRECTED) ---")
# This should now show your manual calculation result (~163M) in the Current 2-3 Mile column
cols = ['Bucket', 'Category', 'Curr_0_2', 'Curr_2_3', 'Curr_3_5', 'Fut_0_2', 'Fut_2_3', 'Fut_3_5']
print(df_exp[cols])

/Users/elyas/Applications/anaconda3/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


--- VERIFIED HOUSEHOLD COUNTS ---
            Curr_0_2  Fut_0_2  Curr_2_3  Fut_2_3  Curr_3_5  Fut_3_5
Households    21,387   21,246    22,539   23,347    67,761   68,446

--- VERIFIED EXPENDITURES (2-3 MILE CORRECTED) ---
  Bucket                    Category    Curr_0_2    Curr_2_3    Curr_3_5  \
0   NG&S                Food at Home 139,369,490 163,892,911 586,849,015   
1   NG&S      Personal Care Products  11,724,578  13,679,762  48,093,942   
2   NG&S       Housekeeping Supplies  16,023,778  18,896,574  67,708,258   
3    F&B         Food Away from Home  78,189,105  91,338,829 328,623,590   
4    F&B         Alcoholic Beverages  12,042,192  14,642,861  54,066,844   
5   GAFO        Apparel and Services  46,968,852  55,237,789 197,380,480   
6   GAFO  Entertainment & Recreation  71,766,922  86,193,633 317,143,341   

      Fut_0_2     Fut_2_3     Fut_3_5  
0 157,583,091 190,703,098 667,403,080  
1  13,263,217  15,947,467  54,688,588  
2  18,118,154  21,983,362  76,978,477  
3  88,408

In [4]:
df_exp.head()

,Curr_0_2,Fut_0_2,Curr_2_3,Fut_2_3,Curr_3_5,Fut_3_5,Bucket,Category
0,"139,369,490","157,583,091","163,892,911","190,703,098","586,849,015","667,403,080",NG&S,Food at Home
1,"11,724,578","13,263,217","13,679,762","15,947,467","48,093,942","54,688,588",NG&S,Personal Care Products
2,"16,023,778","18,118,154","18,896,574","21,983,362","67,708,258","76,978,477",NG&S,Housekeeping Supplies
3,"78,189,105","88,408,554","91,338,829","106,323,914","328,623,590","373,841,225",F&B,Food Away from Home
4,"12,042,192","13,611,226","14,642,861","17,013,868","54,066,844","61,527,422",F&B,Alcoholic Beverages


In [5]:
import pandas as pd
import os

# 1. Define exact paths
base_dir = "../data/Retail"
demand_file = os.path.join(base_dir, "Retail_Demand_Outlook_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2.xlsx")

# 2. Define the Mapping Logic for Buckets
bucket_mapping = {
    'NG&S': ['Food at Home', 'Bakery', 'Meats', 'Dairy', 'Fruits', 'Snacks', 'Personal Care', 'Housekeeping', 'Smoking', 'Drugs'],
    'F&B': ['Food Away from Home', 'Alcoholic Beverages', 'Restaurants'],
    'GAFO': ['Apparel', 'Men\'s', 'Women\'s', 'Children\'s', 'Footwear', 'Watches', 'Jewelry', 'Computer', 'Software', 'Entertainment', 'Fees and Admissions', 'TV', 'Video', 'Audio', 'Pets', 'Toys', 'Sports', 'Furniture', 'Furnishings', 'Textiles', 'Appliances', 'Housewares', 'Luggage']
}

def assign_bucket(cat_name):
    """Maps granular categories to their respective industry buckets."""
    cat_lower = cat_name.lower()
    for bucket, keywords in bucket_mapping.items():
        for kw in keywords:
            if kw.lower() in cat_lower:
                return bucket
    return 'Other'

# 3. Load and Process Data
print("Loading Demand Data...")
df_demand = pd.read_excel(demand_file, header=None, engine='openpyxl')

granular_results = []

# Iterate through the rows (skipping report headers)
for i in range(19, len(df_demand)):
    cat_name = str(df_demand.iloc[i, 0]).strip()
    
    # Filter out empty rows, headers, and footer sources
    if cat_name in ['nan', ''] or 'mile' in cat_name.lower() or 'Source:' in cat_name:
        continue
    
    # Extract numeric values and strip commas
    row_vals = pd.to_numeric(df_demand.iloc[i, 1:].astype(str).str.replace(',', '', regex=False), errors='coerce').dropna().tolist()
    
    # Corrected Indexing logic (Skipping the Growth columns)
    # Index 0: R1_Curr | Index 1: R1_Fut | Index 3: R2_Curr | Index 4: R2_Fut | Index 6: R3_Curr | Index 7: R3_Fut
    if len(row_vals) >= 8:
        granular_results.append({
            'Bucket': assign_bucket(cat_name),
            'Category': cat_name,
            '0-2m_Current_2025': row_vals[0],
            '0-2m_Future_2030': row_vals[1],
            '2-3m_Current_2025': row_vals[3], # Verification: Should be ~163.8M for Food at Home
            '2-3m_Future_2030': row_vals[4],
            '3-5m_Current_2025': row_vals[6],
            '3-5m_Future_2030': row_vals[7]
        })

df_final = pd.DataFrame(granular_results)

# 4. Display & Export
pd.set_option('display.float_format', '{:,.0f}'.format)
print("\n--- GRANULAR RETAIL DATA WITH BUCKET MAPPING ---")
print(df_final.head(15))

# Export to a clean CSV for your Proforma
df_final.to_csv(os.path.join(base_dir, "Granular_Retail_Demand_Analysis.csv"), index=False)

Loading Demand Data...

--- GRANULAR RETAIL DATA WITH BUCKET MAPPING ---
   Bucket                                       Category  0-2m_Current_2025  \
0    GAFO                           Apparel and Services         46,968,852   
1    GAFO                                          Men's          9,227,748   
2    GAFO                                        Women's         15,620,036   
3    GAFO                                     Children's          6,836,102   
4    GAFO                                       Footwear         10,791,770   
5    GAFO                              Watches & Jewelry          3,632,395   
6    GAFO              Apparel Products and Services (1)            860,800   
7    GAFO            Computers and Hardware for Home Use          4,381,080   
8   Other                                Portable Memory             74,237   
9    GAFO                              Computer Software            342,604   
10   GAFO                           Computer Accessories  

In [6]:
df_final.head(60)

,Bucket,Category,0-2m_Current_2025,0-2m_Future_2030,2-3m_Current_2025,2-3m_Future_2030,3-5m_Current_2025,3-5m_Future_2030
0,GAFO,Apparel and Services,"46,968,852","53,110,985","55,237,789","64,283,981","197,380,480","224,518,227"
1,GAFO,Men's,"9,227,748","10,434,001","10,859,665","12,639,777","38,956,403","44,309,429"
2,GAFO,Women's,"15,620,036","17,663,057","19,002,977","22,098,818","68,040,660","77,391,529"
3,GAFO,Children's,"6,836,102","7,730,068","7,571,395","8,811,384","26,599,408","30,241,013"
4,GAFO,Footwear,"10,791,770","12,204,167","12,261,767","14,292,310","43,481,923","49,465,745"
5,GAFO,Watches & Jewelry,"3,632,395","4,106,686","4,521,755","5,254,969","16,563,669","18,857,199"
6,GAFO,Apparel Products and Services (1),"860,800","973,006","1,020,230","1,186,723","3,738,417","4,253,312"
7,GAFO,Computers and Hardware for Home Use,"4,381,080","4,954,698","5,197,343","6,052,767","18,525,189","21,075,426"
8,Other,Portable Memory,"74,237","84,023","90,168","105,216","313,953","357,350"
9,GAFO,Computer Software,"342,604","387,772","409,561","478,629","1,412,563","1,608,321"


In [7]:
import pandas as pd
import os

# 1. SETUP - Define paths
base_dir = "../data/Retail"
demand_file = os.path.join(base_dir, "Retail_Demand_Outlook_4170_E_Ponce_de_Leon_Ave_Clarkston_Georgia_30021_2.xlsx")

# 2. LOAD DATA with error handling for file not found
print("Loading Demand Data...")
if not os.path.exists(demand_file):
    raise FileNotFoundError(f"Input Excel file does not exist: {demand_file}")

df = pd.read_excel(demand_file, header=None, engine="openpyxl")

# 3. HELPER FUNCTIONS
def clean_num(val):
    if pd.isna(val): return 0
    s = str(val).replace(',', '').strip()
    try:
        return float(s)
    except Exception:
        return 0

def get_vals(row_idx):
    """Extracts Current and Future spending for all three rings."""
    try:
        return {
            '0-2m_Current': clean_num(df.iloc[row_idx, 1]) if row_idx < len(df) else 0,
            '0-2m_Future': clean_num(df.iloc[row_idx, 2]) if row_idx < len(df) else 0,
            '2-3m_Current': clean_num(df.iloc[row_idx, 6]) if row_idx < len(df) else 0,
            '2-3m_Future': clean_num(df.iloc[row_idx, 7]) if row_idx < len(df) else 0,
            '3-5m_Current': clean_num(df.iloc[row_idx, 11]) if row_idx < len(df) else 0,
            '3-5m_Future': clean_num(df.iloc[row_idx, 12]) if row_idx < len(df) else 0,
        }
    except Exception:
        return {
            '0-2m_Current': 0,
            '0-2m_Future': 0,
            '2-3m_Current': 0,
            '2-3m_Future': 0,
            '3-5m_Current': 0,
            '3-5m_Future': 0
        }

# 4. DEFINE MAPPING (Excluding summaries to avoid double-counting)
mapping_defs = [
    ('Apparel and Services', 20, list(range(21, 27))),
    ('Computer', 27, list(range(28, 32))),
    ('Education', 32, list(range(33, 35))),
    ('Entertainment & Recreation', 38, list(range(40, 48)) + list(range(49, 62)) + list(range(62, 70))),
    ('Food', 73, [74, 80]), # Food at Home and Food Away
    ('Alcoholic Beverages', 81, [81]),
    ('Financial', 82, list(range(83, 88))),
    ('Health', 88, list(range(89, 94))),
    ('Home', 94, list(range(95, 99))),
    ('Household Furnishings and Equipment', 102, list(range(103, 111))),
    ('Household Operations', 111, list(range(112, 115))),
    ('Housekeeping Supplies (17)', 115, [115]),
    ('Insurance', 116, list(range(117, 121))),
    ('Transportation', 121, list(range(122, 125))),
    ('Travel', 125, list(range(126, 130)))
]

# 5. ROBUSTNESS CHECKS (Verifying math against ESRI parents)
print("\n" + "="*50)
print("    ROBUSTNESS CHECKS (0-2 MILE RING)")
print("="*50)

def verify_category(parent_idx, child_indices, label):
    parent_val = get_vals(parent_idx)['0-2m_Current']
    child_sum = sum(get_vals(i)['0-2m_Current'] for i in child_indices)
    # If parent_val is 0 (NaN in Excel), we check the next header row if available or just verify children
    status = "PASS" if abs(parent_val - child_sum) < 10 or parent_val == 0 else "REVIEW"
    print(f"{label.ljust(30)} | Parent: {parent_val:12,.0f} | Sum: {child_sum:12,.0f} | {status}")

# Check that indices are in range before running checks to prevent index errors
# (Typically these cell indices should match the input, but we'll warn if otherwise)
max_idx = len(df) - 1
def safe_range(start, end):
    return [i for i in range(start, end) if i <= max_idx]
def safe_indices(indices):
    return [i for i in indices if i <= max_idx]

try:
    verify_category(73, safe_indices([74, 80]), "Food Total")
    verify_category(20, safe_range(21, 27), "Apparel Total")
    verify_category(38, safe_indices([39, 48, 62, 63, 64, 65, 66, 67, 68, 69]), "Entertainment Total")
    verify_category(39, safe_range(40, 48), "Fees & Admissions")
    verify_category(48, safe_range(49, 62), "TV/Video/Audio")
    verify_category(111, safe_indices([112, 113, 114, 115]), "Household Operations Total")
except Exception as e:
    print(f"Warning: Robustness check failed due to data shape or unexpected indices. {e}")

# 6. BUILD CLEAN DATASET
final_rows = []
for cat_name, _, indices in mapping_defs:
    for idx in indices:
        # Do not index out of range
        if idx > max_idx:
            continue
        item_name = str(df.iloc[idx, 0]).strip() if idx < len(df) else ""
        # For Food at Home already handled by mapping_defs selection
        row_data = get_vals(idx)
        row_data['Category'] = cat_name
        row_data['Line Item'] = item_name
        final_rows.append(row_data)

df_final = pd.DataFrame(final_rows)
# Reorder for clarity
cols = ['Category', 'Line Item', '0-2m_Current', '0-2m_Future', '2-3m_Current', '2-3m_Future', '3-5m_Current', '3-5m_Future']
df_final = df_final[cols]

# 7. FINAL SUMMARY & EXPORT
print("\n" + "="*50)
print(f"Total Unique Line Items: {len(df_final)}")
print(f"Total 0-2m Current Spend: ${df_final['0-2m_Current'].sum():,.0f}")
print("="*50)

output_csv = os.path.join(base_dir, "Clean_Retail_Expenditures_No_Double_Counting.csv")
try:
    df_final.to_csv(output_csv, index=False)
    print(f"\nSuccess! File saved to: {output_csv}")
except Exception as e:
    print(f"ERROR: Could not write to CSV. {e}")

Loading Demand Data...

    ROBUSTNESS CHECKS (0-2 MILE RING)
Food Total                     | Parent:  217,558,595 | Sum:  217,558,595 | PASS
Apparel Total                  | Parent:   46,968,852 | Sum:   46,968,851 | PASS
Entertainment Total            | Parent:   71,766,922 | Sum:   71,766,922 | PASS
Fees & Admissions              | Parent:   16,225,994 | Sum:   16,225,995 | PASS
TV/Video/Audio                 | Parent:   22,634,169 | Sum:   22,634,168 | PASS
Household Operations Total     | Parent:            0 | Sum:   39,811,842 | PASS

Total Unique Line Items: 81
Total 0-2m Current Spend: $4,505,667,779

Success! File saved to: ../data/Retail/Clean_Retail_Expenditures_No_Double_Counting.csv


In [8]:
from IPython.display import display

# Show all rows of df_final in display
with pd.option_context('display.max_rows', None):
    display(df_final)

,Category,Line Item,0-2m_Current,0-2m_Future,2-3m_Current,2-3m_Future,3-5m_Current,3-5m_Future
0,Apparel and Services,Men's,"9,227,748","10,434,001","10,859,665","12,639,777","38,956,403","44,309,429"
1,Apparel and Services,Women's,"15,620,036","17,663,057","19,002,977","22,098,818","68,040,660","77,391,529"
2,Apparel and Services,Children's,"6,836,102","7,730,068","7,571,395","8,811,384","26,599,408","30,241,013"
3,Apparel and Services,Footwear,"10,791,770","12,204,167","12,261,767","14,292,310","43,481,923","49,465,745"
4,Apparel and Services,Watches & Jewelry,"3,632,395","4,106,686","4,521,755","5,254,969","16,563,669","18,857,199"
5,Apparel and Services,Apparel Products and Services (1),"860,800","973,006","1,020,230","1,186,723","3,738,417","4,253,312"
6,Computer,Computers and Hardware for Home Use,"4,381,080","4,954,698","5,197,343","6,052,767","18,525,189","21,075,426"
7,Computer,Portable Memory,"74,237","84,023","90,168","105,216","313,953","357,350"
8,Computer,Computer Software,"342,604","387,772","409,561","478,629","1,412,563","1,608,321"
9,Computer,Computer Accessories,"371,775","420,402","441,495","514,597","1,588,104","1,808,170"


# Supply Side

In [9]:
# Read in the following dataframes
class_a_retail_df = pd.read_excel("../data/Retail/Class_A_Retail.xlsx")
all_retail_df = pd.read_excel("../data/Retail/All_Retail.xlsx")
comps_retail_df = pd.read_excel("../data/Retail/Comps_Retail.xlsx")

In [10]:
class_a_retail_df.head()

,Period,Inventory Bldgs,Inventory SF,Vacant SF Direct,Vacant SF Sublet,Vacant SF Total,Vacant Percent % Direct,Vacant Percent % Sublet,Vacant Percent % Total,Total Available SF Direct,...,Deliveries Bldgs,Deliveries SF,Under Construction Bldgs,Under Construction SF,All Service Type Rent Direct,All Service Type Rent Sublet,All Service Type Rent Overall,NNN Rent Direct,NNN Rent Sublet,NNN Rent Overall
0,2026 YTD,4,162309,700,-,700,0,-,0,700,...,-,-,-,-,-,-,-,-,-,-
1,2025,4,162309,700,-,700,0,-,0,700,...,-,-,-,-,-,-,-,-,-,-
2,2024,4,162309,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
3,2023,4,162309,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
4,2022,4,162309,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-


In [11]:
#print columns of class_a_retail_df
print(class_a_retail_df.columns)
#print columns of all_retail_df
print(all_retail_df.columns)


Index(['  Period', 'Inventory Bldgs', 'Inventory SF', 'Vacant SF Direct',
       'Vacant SF Sublet', 'Vacant SF Total', 'Vacant Percent % Direct',
       'Vacant Percent % Sublet', 'Vacant Percent % Total',
       'Total Available SF Direct', 'Total Available SF Sublet',
       'Total Available SF Total', 'Total Available Percent % Direct',
       'Total Available Percent % Sublet', 'Total Available Percent % Total',
       'Vacant Available SF Direct', 'Vacant Available SF Sublet',
       'Vacant Available SF Total', 'Vacant Available Percent % Direct',
       'Vacant Available Percent % Sublet', 'Vacant Available Percent % Total',
       'Occupancy SF', 'Occupancy Percent', 'Net Absorption SF Direct',
       'Net Absorption SF Sublet', 'Net Absorption SF Total',
       'Gross Absorption SF Direct', 'Gross Absorption SF Sublet',
       'Gross Absorption SF Total', 'Leasing Activity Deals Direct',
       'Leasing Activity Deals Sublet', 'Leasing Activity Deals Total',
       'Leasing A

In [12]:
import re

def clean_period(period_val):
    # Extract year from values like "2026 YTD" or "2024"
    match = re.search(r'(\d{4})', str(period_val))
    return int(match.group(1)) if match else None

def clean_numeric(val):
    # Remove commas and dash or spaces, handle empty or "-" as NaN
    val = str(val).replace(',', '').strip()
    if val == '-' or val == '':
        return float('nan')
    try:
        return float(val)
    except Exception:
        return float('nan')

def compute_avg_vacancy(df, name="df"):
    # Find the correct column names, accounting for spaces
    period_col = [col for col in df.columns if col.strip().lower() == "period"][0]
    vacancy_col = None
    avail_col = None
    for candidate in ['Vacant Percent % Total', 'Vacant Percent% Total', 'Vacant Percent %Total']:
        matches = [col for col in df.columns if candidate.replace(' ', '').lower() == col.replace(' ', '').lower()]
        if matches:
            vacancy_col = matches[0]
            break
    # Update: look for 'Total Available SF Total' as the available SF column
    for candidate in ['Total Available SF Total']:
        matches = [col for col in df.columns if candidate.replace(' ', '').lower() == col.replace(' ', '').lower()]
        if matches:
            avail_col = matches[0]
            break
    if vacancy_col is None:
        raise ValueError(f"Could not find vacancy column in {name}")
    if avail_col is None:
        # If not found, skip printing Total Available SF and print a message
        print(f"WARNING: Could not find Total Available SF Total column in {name}")
    # Clean columns
    df = df.copy()
    df['Year'] = df[period_col].apply(clean_period)
    df['Vacant_Percent_Total'] = df[vacancy_col].apply(clean_numeric)
    # Multiply by 100 and round to two decimals to be clear
    df['Vacant_Percent_Total'] = (df['Vacant_Percent_Total'] * 100).round(2)
    # If available, clean Total Available SF Total column
    if avail_col:
        df['Total_Available_SF_Clean'] = df[avail_col].apply(clean_numeric)
    # Filter last three years (2023–2026)
    df_recent = df[df['Year'].isin([2024, 2025, 2026, 2023])]
    recent_years = sorted(df_recent['Year'].dropna().unique())[-3:]
    df_recent = df[df['Year'].isin(recent_years)]
    avg_recent = df_recent['Vacant_Percent_Total'].mean()
    avg_total = df['Vacant_Percent_Total'].mean()
    # Get 2026 vacancy if present
    vacancy_2026 = None
    if 2026 in df['Year'].values:
        row_2026 = df[df['Year'] == 2026]
        if not row_2026.empty:
            vacancy_2026 = row_2026['Vacant_Percent_Total'].mean()
    print(f"{name}:")
    print(f"  Vacancy Rate (last 3 years {recent_years}): {avg_recent:.2f}%")
    print(f"  Vacancy Rate (all years): {avg_total:.2f}%")
    if vacancy_2026 is not None and not pd.isna(vacancy_2026):
        print(f"  Vacancy Rate (2026): {vacancy_2026:.2f}%")
    else:
        print(f"  Vacancy Rate (2026): N/A")
    # Print Total Available SF Total if column exists and value is not NaN for 2026
    if avail_col:
        if 2026 in df['Year'].values:
            row_2026 = df[df['Year'] == 2026]
            total_available_2026 = row_2026['Total_Available_SF_Clean'].sum()
            if not pd.isna(total_available_2026):
                print(f"  Total Available SF Total (2026): {int(total_available_2026):,}")
            else:
                print(f"  Total Available SF Total (2026): N/A")
        total_available_sum = df['Total_Available_SF_Clean'].sum()
        print(f"  Total Available SF Total (all years sum): {int(total_available_sum):,}")
    print("")

compute_avg_vacancy(class_a_retail_df, name="Class A Retail")
compute_avg_vacancy(all_retail_df, name="All Retail")

Class A Retail:
  Vacancy Rate (last 3 years [2024, 2025, 2026]): 0.40%
  Vacancy Rate (all years): 4.53%
  Vacancy Rate (2026): 0.40%
  Total Available SF Total (2026): 700
  Total Available SF Total (all years sum): 107,646

All Retail:
  Vacancy Rate (last 3 years [2024, 2025, 2026]): 6.93%
  Vacancy Rate (all years): 7.45%
  Vacancy Rate (2026): 6.30%
  Total Available SF Total (2026): 186,557
  Total Available SF Total (all years sum): 4,887,189



In [13]:
import pandas as pd
pd.set_option('display.max_columns', None)
comps_retail_df.head(20)

,Address,Building Name / Owner,Year Built,Total RBA (SF),Land Area (AC),% Leased,Rent/SF,Primary Use,Anchor Tenants,Average Weighted Rent,Avg Rent-Direct (Retail),Avg Rent-Sublet (Retail),Building Operating Expenses,Building Park,Building Status,Building Tax Expenses,City,Collateral Type,Condo,Construction Begin,Continent,Country,County Name,Coworking Available Space,Cross Street,Developer Name,Direct Available Space,Direct Services,Direct Vacant Space,Features,Fema Flood Zone,FEMA Map Date,FEMA Map Identifier,FIRM ID,FIRM Panel Number,Flood Risk Area,Floodplain Area,For Sale Price Per SF,Fund Name,Has Lab Space,In SFHA,Interest Rate,Interest Rate Type,Land Area (AC).1,Land Area (SF),Leasing Company Address,Leasing Company City State Zip,Leasing Company Contact,Leasing Company Fax,Leasing Company Name,Leasing Company Phone,Loan Type,Maturity Date,Max Floor Contiguous Space,Month Renovated,Number Of Parking Spaces,Number Of Stories,Origination Amount,Origination Date,Originator,Owner Name,Parking Ratio,Percent Leased,Property Address,Property Manager Name,Property Name,Property Type,PropertyID,RBA,Sales Company,Sales Contact,Sales Contact Phone,Smallest Available Space,State,Subcontinent,Sublet Available Space,Sublet Services,Submarket Cluster,Submarket Name,Total Available Space (SF),Year Built.1,Year Renovated,Zip
0,215 Clairemont Ave,Genco Real Estate LLC,1984,5000,0,1,38,Storefront Retail/Office,NaN,38,NaN,NaN,2025 Tax @ $1.8645/sf,NaN,Existing,2025 Tax @ $1.8645/sf,Decatur,Single,No,NaN,Americas,United States,DeKalb,0,NaN,NaN,"1,742",Triple Net,1742,"Air Conditioning, Pylon Sign, Signage, Storage...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,No,NaN,Variable,0,20909,NaN,NaN,Eric Carlton,NaN,Oakhurst Realty Partners,"4,043,714,100",NaN,NaN,"1,742",NaN,30,1,"2,273,800",12/02/2021,ReadyCap Lending LLC,Genco Real Estate LLC,6,65,215 Clairemont Ave,NaN,NaN,Retail (Strip Center),6087336,5000,NaN,NaN,NaN,"1,742",GA,North America,0,NaN,Dekalb,Decatur/East Atl,"1,742",1984,NaN,30030
1,619-647 E College Ave,"Meghan Byrne, LLC",1945/04,24000,3,1,26,Freestanding,NaN,24,26,NaN,2025 Tax @ $0.3127/sf,East Decatur Station,Existing,2025 Tax @ $0.3127/sf,Decatur,NaN,No,NaN,Americas,United States,DeKalb,0,NaN,NaN,"4,950",Triple Net,4950,"Property Manager on Site, Pylon Sign, Restaura...","Area of moderate flood hazard, usually the are...",08/15/2019,13089C0068K,13089C,0068K,Moderate to Low Risk Areas,100-year and 500-year Floodplain,NaN,NaN,False,No,NaN,NaN,3,110642,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"2,200",NaN,65,2,NaN,NaN,NaN,"Meghan Byrne, LLC",3,79,619-647 E College Ave,NaN,NaN,Retail,923164,24000,NaN,NaN,NaN,"1,000",GA,North America,0,NaN,Dekalb,Decatur/East Atl,"4,950",1945,"2,004",30030
2,119-123 E Court Sq,East Court Square II LLC,1927,14448,0,1,36,Storefront Retail/Office,NaN,36,NaN,NaN,2025 Tax @ $0.7272/sf,NaN,Existing,2025 Tax @ $0.7272/sf,Decatur,NaN,No,Jul 1926,Americas,United States,DeKalb,0,NaN,NaN,"1,629",Full Service,2509,"24 Hour Access, Air Conditioning, Tenant Contr...","Area of moderate flood hazard, usually the are...",08/15/2019,13089C0068K,13089C,0068K,Moderate to Low Risk Areas,100-year and 500-year Floodplain,NaN,NaN,False,No,NaN,NaN,0,7224,NaN,NaN,Susan Poole,NaN,"Corporate Property Advisors, Inc.","4,042,755,741",NaN,NaN,"1,629",NaN,36,2,NaN,NaN,NaN,East Court Square II LLC,2,89,119-123 E Court Sq,"Corporate Property Advisors, Inc.",NaN,Retail,437449,14448,NaN,NaN,NaN,408,GA,North America,0,NaN,Dekalb,Decatur/East Atl,"1,629",1927,NaN,30030-2521
3,929 Main St,—,1929,4052,0,1,20,Restaurant,NaN,20,20,NaN,2025 Tax @ $0.3654/sf,NaN,Existing,2025 Tax @ $0.3654/sf,Stone Mountain,NaN,No,NaN,Americas,United States,DeKalb,0,NaN,NaN,"1,730",Modified Gross,1730,"Corner Lot, Restaurant",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,No,NaN,NaN,0,3049,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"1,730",NaN,NaN,2,NaN,NaN,NaN,Cloud Street Development Inc,0,57,929 Main St,NaN,NaN,Retail,13829974,4052,NaN,NaN,NaN,"1,730",GA,North America,0,NaN,Dekal

In [14]:
# Clean column names to remove leading/trailing spaces
comps_retail_df.columns = comps_retail_df.columns.str.strip()

# Find likely column names
def find_column(df, options):
    for opt in options:
        # This replaces spaces and lowercases for a "fuzzy" match
        matches = [col for col in df.columns if col.replace(" ", "").lower() == opt.replace(" ", "").lower()]
        if matches:
            return matches[0]
    raise ValueError(f"Column not found for options: {options}")

# Updated search options to match your specific table headers
rba_col = find_column(comps_retail_df, ['RBA', 'Total RBA (SF)', 'Total RBA'])
rent_per_sf_col = find_column(comps_retail_df, ['Rent/SF', 'Rent Per SF', 'Asking Rent/SF'])
avg_weighted_rent_col = find_column(comps_retail_df, ['Average Weighted Rent', 'Avg Weighted Rent'])
direct_avail_col = find_column(comps_retail_df, ['Direct Available Space', 'Direct Avail'])
total_avail_col = find_column(comps_retail_df, ['Total Available Space (SF)', 'Total Available SF', 'Total Avail'])
percent_leased_col = find_column(comps_retail_df, ['Percent Leased', '% Leased', 'Leased %'])

# Convert to numeric, handling commas and strings like "—"
cols_to_fix = [rba_col, rent_per_sf_col, avg_weighted_rent_col, direct_avail_col, total_avail_col, percent_leased_col]
for col in cols_to_fix:
    comps_retail_df[col] = pd.to_numeric(comps_retail_df[col].astype(str).str.replace(',', '').replace('—', ''), errors='coerce')

# Calculations
sum_rba = comps_retail_df[rba_col].sum()
avg_rent_sf = comps_retail_df[rent_per_sf_col].mean()
avg_weighted_rent = comps_retail_df[avg_weighted_rent_col].mean()
sum_direct_avail = comps_retail_df[direct_avail_col].sum()
sum_total_avail = comps_retail_df[total_avail_col].sum()

# Vacancy Rate Formulas
comps_retail_df['Vacancy Rate'] = (comps_retail_df[total_avail_col] / comps_retail_df[rba_col]) * 100
avg_vacancy_rate = comps_retail_df['Vacancy Rate'].mean()

comps_retail_df['Vacancy Rate 2'] = 100 - comps_retail_df[percent_leased_col]
avg_vacancy_rate_2 = comps_retail_df['Vacancy Rate 2'].mean()

# Output
print(f"Sum of RBA: {sum_rba:,.0f}")
print(f"Average of Rent/SF: ${avg_rent_sf:,.2f}")
print(f"Average of Average Weighted Rent: ${avg_weighted_rent:,.2f}")
print(f"Sum of Direct Available Space: {sum_direct_avail:,.0f}")
print(f"Sum of Total Available SF: {sum_total_avail:,.0f}")
print(f"Average Vacancy Rate (Total Avail / RBA): {avg_vacancy_rate:.2f}%")
print(f"Average Vacancy Rate 2 (100 - % Leased): {avg_vacancy_rate_2:.2f}%")

Sum of RBA: 151,013
Average of Rent/SF: $29.86
Average of Average Weighted Rent: $29.96
Sum of Direct Available Space: 18,232
Sum of Total Available SF: 18,232
Average Vacancy Rate (Total Avail / RBA): 24.39%
Average Vacancy Rate 2 (100 - % Leased): 9.48%


In [15]:
# Fill empty available space with 0 so they are included in the average
comps_retail_df[total_avail_col] = comps_retail_df[total_avail_col].fillna(0)

# Now calculate the rates
comps_retail_df['Vacancy Rate'] = (comps_retail_df[total_avail_col] / comps_retail_df[rba_col]) * 100
print(f"Simple Average Vacancy: {comps_retail_df['Vacancy Rate'].mean():.2f}%")

# Calculate the Portfolio (Weighted) Vacancy
portfolio_vacancy = (sum_total_avail / sum_rba) * 100
print(f"True Portfolio Vacancy: {portfolio_vacancy:.2f}%")

Simple Average Vacancy: 9.48%
True Portfolio Vacancy: 12.07%
